# Day 30/30 — Final Capstone: Citizen Grievance Intelligence
**30-Day Data Analytics Cookbook | From Foundations to Real-World Impact**

Build a reproducible decision-support project: data quality, KPIs, diagnosis, forecasting, dashboard, executive reporting and portfolio development. All records are synthetic; no actual GCC performance is represented.

## Business question
Where should a municipal administration investigate unresolved workloads, and what complaint intake might it expect next month? Use a fixed reporting cutoff, 31 December 2025. Closure counts refer to closure events; cohort resolution refers to complaints registered in the selected period and closed by the cutoff. No reopening is simulated.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display
rng = np.random.default_rng(30)
out = Path('day30_outputs'); out.mkdir(exist_ok=True)
cutoff = pd.Timestamp('2025-12-31 23:59:59')
months = pd.date_range('2023-01-01', '2025-12-01', freq='MS')
rows = []
for i, month in enumerate(months):
    volume = int(450 + 5*i + 90*np.sin(2*np.pi*(month.month-8)/12) + rng.normal(0,25))
    for _ in range(volume):
        zone = rng.choice(['Zone A','Zone B','Zone C','Zone D','Zone E'], p=[.25,.25,.20,.17,.13])
        category = rng.choice(['Roads','Sanitation','Drainage','Streetlights'], p=[.25,.30,.25,.20])
        registered = month + pd.Timedelta(days=int(rng.integers(0,month.days_in_month)))
        duration = int(rng.gamma(2, 4 if zone != 'Zone B' else 8))+1
        closed = registered + pd.Timedelta(days=duration)
        if closed > cutoff or rng.random() < .035: closed = pd.NaT
        rows.append((f'G{len(rows)+1:06}',registered,zone,category,closed))
raw = pd.DataFrame(rows,columns=['complaint_id','registered','zone','category','closed'])
raw = pd.concat([raw,raw.iloc[:20]],ignore_index=True)
raw.loc[30,'registered'] = pd.NaT
raw.loc[31,'closed'] = pd.Timestamp('2020-01-01')
raw.to_csv(out/'raw.csv',index=False)
print('Raw rows:',len(raw))

Raw rows: 19373


## Data dictionary and quality rules
`complaint_id`: unique synthetic case identifier; `registered`: intake date; `zone`: fictional administrative area; `category`: service type; `closed`: closure date or missing for open cases. Duplicate IDs are removed (identical copies in this simulation). Invalid dates are quarantined, not treated as open cases. In production, conflicting duplicates require investigation.

In [2]:
df = raw.drop_duplicates('complaint_id').copy()
invalid = df.registered.isna() | (df.registered > cutoff) | (df.closed.notna() & ((df.closed < df.registered) | (df.closed > cutoff)))
rejected = df.loc[invalid].copy()
df = df.loc[~invalid].copy()
df['resolved'] = df.closed.notna()
df['resolution_days'] = (df.closed-df.registered).dt.days
df['open_age_days'] = (cutoff.normalize()-df.registered).dt.days.where(~df.resolved)
df['overdue'] = (~df.resolved) & (df.open_age_days > 14)
audit = pd.Series({'raw':len(raw),'duplicate_rows':len(raw)-raw.complaint_id.nunique(),'invalid_unique_rows':len(rejected),'clean':len(df)})
assert audit.raw == audit.duplicate_rows + audit.invalid_unique_rows + audit.clean
assert df.complaint_id.is_unique
assert df.resolution_days.dropna().ge(0).all()
display(audit)
audit.to_csv(out/'quality_audit.csv'); rejected.to_csv(out/'rejected.csv',index=False); df.to_csv(out/'clean.csv',index=False)

raw                    19373
duplicate_rows            20
invalid_unique_rows        2
clean                  19351
dtype: int64

## KPI analysis
The 14-day overdue threshold is an illustrative analytical rule, not an official service-level agreement. Resolution duration excludes open cases and therefore does not describe their waiting time.

In [3]:
kpis = pd.Series({'registered':len(df),'closed':int(df.resolved.sum()),'open':int((~df.resolved).sum()),'cohort_resolution_pct':100*df.resolved.mean(),'median_closed_days':df.resolution_days.median(),'overdue_open':int(df.overdue.sum())})
assert kpis.registered == kpis.closed + kpis.open
display(kpis.round(2)); kpis.to_csv(out/'kpis.csv')
zone = df.groupby('zone').agg(received=('complaint_id','size'),closed=('resolved','sum'),overdue=('overdue','sum'),median_days=('resolution_days','median'))
zone['open'] = zone.received-zone.closed
zone['resolution_pct'] = 100*zone.closed/zone.received
zone = zone.sort_values('overdue',ascending=False)
category = df.groupby('category').agg(received=('complaint_id','size'),closed=('resolved','sum'),overdue=('overdue','sum'))
zone.to_csv(out/'zones.csv'); category.to_csv(out/'categories.csv'); display(zone.round(2)); display(category)

registered               19351.00
closed                   18434.00
open                       917.00
cohort_resolution_pct       95.26
median_closed_days           8.00
overdue_open               729.00
dtype: float64

,received,closed,overdue,median_days,open,resolution_pct
zone,,,,,,
Zone B,4850,4589,205,14.0,261,94.62
Zone A,4750,4554,156,7.0,196,95.87
Zone C,3871,3701,135,7.0,170,95.61
Zone D,3227,3069,128,7.0,158,95.10
Zone E,2653,2521,105,7.0,132,95.02


,received,closed,overdue
category,,,
Drainage,4925,4697,177
Roads,4806,4577,183
Sanitation,5751,5468,229
Streetlights,3869,3692,140


In [4]:
monthly = pd.DataFrame({'received':df.groupby(df.registered.dt.to_period('M')).size(),'closed':df.loc[df.resolved].groupby(df.closed.dt.to_period('M')).size()}).reindex(months.to_period('M')).fillna(0).astype(int)
monthly['net_gap'] = monthly.received-monthly.closed
monthly['backlog'] = monthly.net_gap.cumsum()
assert monthly.backlog.iloc[-1] == kpis.open
monthly.to_csv(out/'monthly.csv'); display(monthly.tail())

,received,closed,net_gap,backlog
2025-08,606,567,39,804
2025-09,663,588,75,879
2025-10,662,684,-22,857
2025-11,694,641,53,910
2025-12,639,632,7,917


## Management dashboard
Use separate views for intake/closure events, accumulated backlog, and overdue cases. The backlog identity holds here because the synthetic system begins with zero open cases and has no reopened or cancelled cases.

In [5]:
def save_plot(ax,name):
    ax.figure.tight_layout(); ax.figure.savefig(out/name,dpi=150,bbox_inches='tight'); plt.show(); plt.close(ax.figure)
ax = monthly[['received','closed']].plot(figsize=(10,4),title='Monthly intake and closure events',ylabel='Cases'); save_plot(ax,'monthly.png')
ax = monthly.backlog.plot(figsize=(10,4),title='Outstanding workload at month end',ylabel='Open cases'); save_plot(ax,'backlog.png')
ax = zone.overdue.sort_values().plot.barh(title='Overdue open cases by zone',xlabel='Cases'); save_plot(ax,'overdue.png')
heat = df.pivot_table(index='zone',columns='category',values='overdue',aggfunc='sum',fill_value=0)
fig,ax=plt.subplots(figsize=(8,4)); im=ax.imshow(heat.values); ax.set_xticks(range(len(heat.columns)),heat.columns,rotation=25); ax.set_yticks(range(len(heat.index)),heat.index); fig.colorbar(im,ax=ax,label='Overdue cases'); ax.set_title('Overdue workload by zone and category'); save_plot(ax,'heatmap.png')

/tmp/ipykernel_510/2026886810.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.figure.tight_layout(); ax.figure.savefig(out/name,dpi=150,bbox_inches='tight'); plt.show(); plt.close(ax.figure)
/tmp/ipykernel_510/2026886810.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.figure.tight_layout(); ax.figure.savefig(out/name,dpi=150,bbox_inches='tight'); plt.show(); plt.close(ax.figure)


/tmp/ipykernel_510/2026886810.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.figure.tight_layout(); ax.figure.savefig(out/name,dpi=150,bbox_inches='tight'); plt.show(); plt.close(ax.figure)
/tmp/ipykernel_510/2026886810.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.figure.tight_layout(); ax.figure.savefig(out/name,dpi=150,bbox_inches='tight'); plt.show(); plt.close(ax.figure)


## Forecasting and evaluation
Compare naive, seasonal naive and trailing three-month mean using the last 12 months as a chronological holdout. Each forecast is one-step-ahead using only observations available before that month. Choose the model using a separate validation window (months 13–24), then report its holdout performance. This is not a fixed-origin 12-month forecast.

In [6]:
y = monthly.received.astype(float)
pred = pd.DataFrame({'actual':y,'naive':y.shift(1),'seasonal_naive':y.shift(12),'moving_average':y.shift(1).rolling(3).mean()})
validation = pred.iloc[12:24].dropna(); test = pred.iloc[24:].dropna()
models = ['naive','seasonal_naive','moving_average']
def metrics(frame):
    return pd.DataFrame({name:{'MAE':mean_absolute_error(frame.actual,frame[name]),'RMSE':np.sqrt(mean_squared_error(frame.actual,frame[name])),'bias_pred_minus_actual':(frame[name]-frame.actual).mean()} for name in models}).T
val_metrics=metrics(validation); chosen=val_metrics.MAE.idxmin(); test_metrics=metrics(test)
display(val_metrics.round(2)); display(test_metrics.round(2)); print('Selected on validation:',chosen)
test_metrics.to_csv(out/'holdout_metrics.csv')
next_predictions={'naive':y.iloc[-1],'seasonal_naive':y.iloc[-12],'moving_average':y.iloc[-3:].mean()}
next_forecast=next_predictions[chosen]
print('January 2026 intake point forecast:',round(next_forecast))
ax=test[['actual',chosen]].plot(marker='o',title='Holdout: actual vs selected model',ylabel='Complaints'); save_plot(ax,'forecast.png')

,MAE,RMSE,bias_pred_minus_actual
naive,32.75,37.80,-10.08
seasonal_naive,62.75,69.76,-57.42
moving_average,55.17,59.46,-15.56


,MAE,RMSE,bias_pred_minus_actual
naive,36.92,40.33,1.92
seasonal_naive,58.33,65.64,-54.50
moving_average,56.56,62.65,-0.22


Selected on validation: naive
January 2026 intake point forecast: 639


/tmp/ipykernel_510/2026886810.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.figure.tight_layout(); ax.figure.savefig(out/name,dpi=150,bbox_inches='tight'); plt.show(); plt.close(ax.figure)


## Executive briefing and operational plan
Recommendations are hypotheses for investigation, not proof of causation. A point forecast has no calibrated uncertainty interval; use it as a planning input, not a staffing guarantee.

In [7]:
priority=zone.index[0]
brief=f"""# Citizen Grievance Intelligence — Executive Brief
Reporting cutoff: {cutoff.date()} | Synthetic demonstration only

- Registered: {int(kpis.registered):,}; closed: {int(kpis.closed):,}; open: {int(kpis.open):,}.
- Cohort resolution: {kpis.cohort_resolution_pct:.2f}%; overdue open cases: {int(kpis.overdue_open):,}.
- Priority investigation: {priority}, with {int(zone.loc[priority,'overdue'])} overdue cases.
- Next-month intake forecast: {next_forecast:.0f}; selected model: {chosen}; holdout MAE: {test_metrics.loc[chosen,'MAE']:.1f}.

## Action plan
1. Within one week, review overdue cases and category-level bottlenecks in {priority}.
2. Assign accountable service leads and assess whether additional field capacity is justified.
3. Monitor open and overdue cases weekly; illustrative target: reduce overdue workload by 20% in 30 days without compromising closure quality.

Limitations: synthetic data, no reopening, no official SLA, no causal inference, no forecast interval.
"""
(out/'executive_summary.md').write_text(brief,encoding='utf-8'); print(brief)

# Citizen Grievance Intelligence — Executive Brief
Reporting cutoff: 2025-12-31 | Synthetic demonstration only

- Registered: 19,351; closed: 18,434; open: 917.
- Cohort resolution: 95.26%; overdue open cases: 729.
- Priority investigation: Zone B, with 205 overdue cases.
- Next-month intake forecast: 639; selected model: naive; holdout MAE: 36.9.

## Action plan
1. Within one week, review overdue cases and category-level bottlenecks in Zone B.
2. Assign accountable service leads and assess whether additional field capacity is justified.
3. Monitor open and overdue cases weekly; illustrative target: reduce overdue workload by 20% in 30 days without compromising closure quality.

Limitations: synthetic data, no reopening, no official SLA, no causal inference, no forecast interval.



## Portfolio roadmap — next 90 days
| Period | Focus | Evidence to publish |
|---|---|---|
| Days 1–30 | Advanced SQL, Pandas, Git, dashboard design | Reproducible KPI dashboard and data dictionary |
| Days 31–60 | Forecasting, classification, validation | Two model comparisons with holdout results |
| Days 61–90 | APIs, automation, deployment, governance | Deployed demo, tests, monitoring and documentation |

### Portfolio checklist
- Business question and stakeholder context
- Synthetic data and quality audit
- Reproducible notebook and dependencies
- Visuals, metrics, executive summary and limitations
- Clear GitHub README and concise project presentation

### Exercises
1. Change the overdue threshold and explain the sensitivity of zone rankings.
2. Compare category-specific forecasts with the aggregate forecast.
3. Add a closure-quality metric and explain why closure volume alone is insufficient.
4. Convert the dashboard to Streamlit using the exported clean dataset.

**Data analytics is not about dashboards. It is about decisions.**

[Complete cookbook](https://github.com/krishnakumarimv/30-Days-Data-Analytics-Cookbook)